## SetFit Binary Classification Training

#### Load in Python Libraries

In [15]:
from datasets import Dataset, DatasetDict
from setfit import SetFitModel, Trainer, TrainingArguments, sample_dataset
import pandas as pd
import mlflow
import os
from mlflow.pyfunc import PythonModel
mlflow.transformers.autolog()

#### Set Experiment

In [2]:
mlflow.set_experiment("adeptID")

<Experiment: artifact_location='file:///c:/Users/jvhua/OneDrive/Desktop/ISYE-CSE-MGT-6748-Group-1/02_preprocess/mlruns/418082982984917409', creation_time=1719438288344, experiment_id='418082982984917409', last_update_time=1719438288344, lifecycle_stage='active', name='adeptID', tags={}>

#### Load in Data that was labeled in excel. 
    * Two variations
        * 300 labeled dataset 
        * 3500 labeled dataset

In [3]:
df = pd.read_excel(r"C:\Users\jvhua\OneDrive\Desktop\ISYE-CSE-MGT-6748-Group-1\data\set-fit-datav2.xlsx")
df.columns
df = df[['id', 'text', 'label']].reset_index(drop = True)

#### Change to Hugging Face Format

In [4]:
dataset = Dataset.from_pandas(df)

train_val_test_split = dataset.train_test_split(test_size=0.3)

train_val_split = train_val_test_split['train'].train_test_split(test_size=0.5)

dataset_dict = DatasetDict({
    'train': train_val_split['train'],
    'validation': train_val_split['test'],
    'test': train_val_test_split['test']
})

In [5]:
dataset_dict

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'label'],
        num_rows: 1224
    })
    validation: Dataset({
        features: ['id', 'text', 'label'],
        num_rows: 1225
    })
    test: Dataset({
        features: ['id', 'text', 'label'],
        num_rows: 1050
    })
})

In [6]:
train_dataset = dataset_dict['train']
eval_dataset = dataset_dict["validation"]
test_dataset = dataset_dict["test"]

#### Load in SetFit Model to Train

In [7]:
model = SetFitModel.from_pretrained(
    "sentence-transformers/paraphrase-mpnet-base-v2",
    labels=[0, 1],
)

c:\Users\jvhua\miniconda3\envs\langchain\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.


#### Set Model Arguements

In [11]:
args = TrainingArguments(
    batch_size=8,
    num_epochs=5,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    seed = 7,
    warmup_proportion = 0.1,
    num_iterations=5,
    max_length = 512,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    metric="accuracy",
    column_mapping={"text": "text", "label": "label"}
)

Applying column mapping to the training dataset
Applying column mapping to the evaluation dataset


Map:   0%|          | 0/1224 [00:00<?, ? examples/s]

#### Train Model

In [12]:
with mlflow.start_run(run_name = "setfit-preprocess-model-train"):
    trainer.train()
    metrics = trainer.evaluate(test_dataset)
    mlflow.log_metric('accuracy',  metrics['accuracy'])
    print(metrics)

***** Running training *****
  Num unique pairs = 12240
  Batch size = 8
  Num epochs = 5
  Total optimization steps = 7650


  0%|          | 0/7650 [00:00<?, ?it/s]

  0%|          | 0/7650 [00:00<?, ?it/s]

{'embedding_loss': 0.3255, 'learning_rate': 2.6143790849673203e-08, 'epoch': 0.0}
{'embedding_loss': 0.3934, 'learning_rate': 1.3071895424836604e-06, 'epoch': 0.03}
{'embedding_loss': 0.2528, 'learning_rate': 2.6143790849673208e-06, 'epoch': 0.07}
{'embedding_loss': 0.2546, 'learning_rate': 3.92156862745098e-06, 'epoch': 0.1}
{'embedding_loss': 0.2683, 'learning_rate': 5.2287581699346416e-06, 'epoch': 0.13}
{'embedding_loss': 0.2329, 'learning_rate': 6.535947712418301e-06, 'epoch': 0.16}
{'embedding_loss': 0.2699, 'learning_rate': 7.84313725490196e-06, 'epoch': 0.2}
{'embedding_loss': 0.1307, 'learning_rate': 9.150326797385621e-06, 'epoch': 0.23}
{'embedding_loss': 0.232, 'learning_rate': 1.0457516339869283e-05, 'epoch': 0.26}
{'embedding_loss': 0.0697, 'learning_rate': 1.1764705882352942e-05, 'epoch': 0.29}
{'embedding_loss': 0.0927, 'learning_rate': 1.3071895424836602e-05, 'epoch': 0.33}
{'embedding_loss': 0.2142, 'learning_rate': 1.4379084967320263e-05, 'epoch': 0.36}
{'embedding_lo

  0%|          | 0/1532 [00:00<?, ?it/s]

...

{'eval_embedding_loss': 0.2478, 'learning_rate': 0.0, 'epoch': 5.0}


Loading best SentenceTransformer model from step 1530.


{'train_runtime': 3923.0498, 'train_samples_per_second': 15.6, 'train_steps_per_second': 1.95, 'epoch': 5.0}


Applying column mapping to the evaluation dataset
***** Running evaluation *****


{'accuracy': 0.8590476190476191}


#### Save Model

    * Save model as folder 
    * Save model through mlFlow

In [16]:
model_save_name = 'sefit_model_v6'

folder_path = f"C:\\Users\\jvhua\\OneDrive\\Desktop\\ISYE-CSE-MGT-6748-Group-1\\{model_save_name}"
if not os.path.exists(folder_path):
    os.makedirs(folder_path)

##### Save model as Model Folder

In [17]:
save_directory = folder_path
trainer.model._save_pretrained(save_directory=save_directory)

##### Save model as mlflow model 

In [18]:
class SetFitWrapper(PythonModel):

    def load_context(self, context):
        self.model = SetFitModel.from_pretrained(context.artifacts['model'])

    def predict(self, context, model_input):
        return self.model.predict(model_input)

In [19]:
with mlflow.start_run(run_name = "setfit-preprocess-model-train"):
    mlflow.pyfunc.log_model(
        artifact_path='setfit_model',
        python_model=SetFitWrapper(),
        artifacts={'model': save_directory}
    )
    mlflow.log_param('model_name', model_save_name)
mlflow.end_run()

c:\Users\jvhua\miniconda3\envs\langchain\Lib\importlib\__init__.py:126: DeprecationWarning: The distutils package is deprecated and slated for removal in Python 3.12. Use setuptools or check PEP 632 for potential alternatives
  return _bootstrap._gcd_import(name[level:], package, level)
